<a href="https://colab.research.google.com/github/Vishali-Vivek006/Engineering-employability-prediction/blob/main/notebooks/Untitled11.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

import joblib

model = joblib.load("employability_model.joblib")
preprocessor = joblib.load("preprocessor.joblib")

print("Model type:", type(model).__name__)
print("Preprocessor type:", type(preprocessor).__name__)

if hasattr(preprocessor, "feature_names_in_"):
    print("\nRaw training input features:")
    print(list(preprocessor.feature_names_in_))
else:
    print("\nThe preprocessor does not expose feature_names_in_.")

if hasattr(model, "n_features_in_"):
    print("\nModel input feature count:", model.n_features_in_)

if hasattr(model, "named_steps"):
    print("\nModel pipeline steps:")
    print(list(model.named_steps.keys()))


Model type: Pipeline
Preprocessor type: ColumnTransformer

Raw training input features:
['Education_Level', 'UG_Degree', 'PG_Specialization', 'PG_Required_For_Target_Role', 'Relevant_Degree_Required', 'Branch', 'Year', 'CGPA', 'Attendance_Percentage', 'Backlogs', 'Core_Subject_Score', 'Technical_Subject_Score', 'Aptitude_Score', 'C', 'Cpp', 'Java', 'Python', 'JavaScript', 'TypeScript', 'CSharp', 'R', 'Go', 'Kotlin', 'PHP', 'Ruby', 'Swift', 'Dart', 'MATLAB', 'DSA', 'OOP', 'DBMS', 'Operating_Systems', 'Computer_Networks', 'Computer_Architecture', 'Software_Engineering', 'Computer_Security', 'Cloud_Computing', 'Distributed_Systems', 'Compiler_Design', 'Theory_of_Computation', 'HTML', 'CSS', 'React', 'Angular', 'Vue', 'NodeJS', 'ExpressJS', 'NextJS', 'Spring_Boot', 'Django', 'Flask', 'REST_API', 'GraphQL', 'Web_Development', 'SQL', 'MySQL', 'PostgreSQL', 'Oracle', 'MongoDB', 'Redis', 'Firebase', 'NumPy', 'Pandas', 'Matplotlib', 'Scikit_Learn', 'TensorFlow', 'PyTorch', 'Machine_Learning', '

In [2]:

import os
import json

print("=" * 60)
print("CHECKING UPLOADED PROJECT FILES")
print("=" * 60)

for filename in sorted(os.listdir("/content")):
    print(filename)

print("\n" + "=" * 60)
print("SEARCHING FOR RESUME EXTRACTION OUTPUT")
print("=" * 60)

json_files = [
    f for f in os.listdir("/content")
    if f.lower().endswith(".json")
]

if json_files:
    for filename in json_files:
        print("\nJSON file found:", filename)

        try:
            with open(
                os.path.join("/content", filename),
                "r",
                encoding="utf-8"
            ) as file:
                data = json.load(file)

            if isinstance(data, dict):
                print("Available fields:", list(data.keys()))
                print("Preview:")
                print(json.dumps(data, indent=2)[:2500])

        except Exception as e:
            print("Could not read as JSON:", e)
else:
    print("No JSON file found in /content.")
    print("The uploaded files may be in another Colab session.")


CHECKING UPLOADED PROJECT FILES
.config
employability_model.joblib
final_employability_model.joblib
preprocessor.joblib
resume_extraction_output (1).json
resume_extraction_output.json
sample_data

SEARCHING FOR RESUME EXTRACTION OUTPUT

JSON file found: resume_extraction_output (1).json
Available fields: ['resume_filename', 'education_mentions', 'cgpa_mentions', 'technical_skills', 'skills_by_category', 'project_section', 'internship_section', 'certification_section']
Preview:
{
  "resume_filename": "Resume_Harshita.pdf",
  "education_mentions": [
    "B. E"
  ],
  "cgpa_mentions": [],
  "technical_skills": [
    "CSS",
    "HTML",
    "JavaScript",
    "Python"
  ],
  "skills_by_category": {
    "Programming": [
      "Python",
      "JavaScript"
    ],
    "Web Development": [
      "HTML",
      "CSS"
    ],
    "Database": [],
    "CS Fundamentals": [],
    "AI ML Data": [],
    "Cloud DevOps Tools": []
  },
  "project_section": "",
  "internship_section": "CNC Machining \u2013 Rou

In [3]:

# ============================================================
# STEP 3C: EXISTING RESUME -> EMPLOYABILITY MODEL
# ============================================================

import os
import re
import json
import joblib
import numpy as np
import pandas as pd

BASE = "/content"

# ------------------------------------------------------------
# 1. Load the existing extraction output and saved artifacts
# ------------------------------------------------------------
resume_path = os.path.join(BASE, "resume_extraction_output.json")
model_path = os.path.join(BASE, "final_employability_model.joblib")
preprocessor_path = os.path.join(BASE, "preprocessor.joblib")

for path in [resume_path, model_path, preprocessor_path]:
    if not os.path.exists(path):
        raise FileNotFoundError(f"Required file not found: {path}")

with open(resume_path, "r", encoding="utf-8") as f:
    resume = json.load(f)

model = joblib.load(model_path)
preprocessor = joblib.load(preprocessor_path)

# Raw columns used when the preprocessor was fitted
expected_columns = list(preprocessor.feature_names_in_)

print("=" * 65)
print("RESUME-TO-MODEL INTEGRATION")
print("=" * 65)
print("Resume:", resume.get("resume_filename", "Unknown"))
print("Extracted skills:", resume.get("technical_skills", []))
print("Expected raw input columns:", len(expected_columns))

# ------------------------------------------------------------
# 2. Create a raw input row
#    Missing values remain missing unless safely mapped below.
# ------------------------------------------------------------
row = {col: np.nan for col in expected_columns}

# Categorical inputs: Unknown is handled by the fitted encoder
categorical_columns = {
    "Education_Level",
    "UG_Degree",
    "PG_Specialization",
    "Branch"
}

for col in categorical_columns:
    if col in row:
        row[col] = "Unknown"

# Skills explicitly found in the resume
resume_skills = {
    str(skill).strip().lower()
    for skill in resume.get("technical_skills", [])
}

# Aliases to match resume terminology with dataset column names
skill_aliases = {
    "c++": "cpp",
    "node.js": "nodejs",
    "machine learning": "machinelearning",
    "deep learning": "deeplearning",
    "computer networks": "computernetworks",
    "data structures": "datastructures"
}

def normalize_skill(value):
    value = str(value).strip().lower()
    value = skill_aliases.get(value, value)
    return re.sub(r"[^a-z0-9]", "", value)

normalized_resume_skills = {
    normalize_skill(skill) for skill in resume_skills
}

# Parse CGPA only when a value was actually extracted
cgpa = np.nan
for mention in resume.get("cgpa_mentions", []):
    match = re.search(
        r"(?:CGPA|GPA)\s*[:\-]?\s*(\d+(?:\.\d+)?)",
        str(mention),
        flags=re.IGNORECASE
    )
    if match:
        cgpa = float(match.group(1))
        break

if "CGPA" in row and not np.isnan(cgpa):
    row["CGPA"] = cgpa

# Map detected binary technical skills.
# Do not mark unmentioned skills as confirmed skills.
for col in expected_columns:
    normalized_col = normalize_skill(col)

    if normalized_col in normalized_resume_skills:
        row[col] = 1

# A missing resume mention is not proof the student has no skill.
# Therefore, leave all other fields missing for the fitted
# preprocessor's imputation rather than fabricating values.

# ------------------------------------------------------------
# 3. Show what was and was not mapped
# ------------------------------------------------------------
input_df = pd.DataFrame([row], columns=expected_columns)

print("\nExtracted CGPA:", cgpa if not np.isnan(cgpa) else "Not found")
print("Number of extracted skills:", len(resume_skills))

mapped = [
    col for col in expected_columns
    if pd.notna(input_df.iloc[0][col])
]

missing = [
    col for col in expected_columns
    if pd.isna(input_df.iloc[0][col])
]

print("\nFields mapped before preprocessing:", len(mapped))
print("Fields still missing:", len(missing))
print("\nMapped fields:")
print(mapped)
print("\nMissing fields (first 50):")
print(missing[:50])

# ------------------------------------------------------------
# 4. Stop if required role/academic information is unavailable
#    This prevents a misleading prediction from a mostly empty row.
# ------------------------------------------------------------
important_fields = [
    "Education_Level",
    "UG_Degree",
    "Branch",
    "Year",
    "CGPA",
    "Internship",
    "Internship_Duration_Months",
    "Project_Count",
    "Relevant_Project_Count",
    "Backlogs",
    "Core_Subject_Score",
    "Technical_Subject_Score",
    "Aptitude_Score"
]

unavailable_important = [
    col for col in important_fields
    if col in input_df.columns and pd.isna(input_df.iloc[0][col])
]

print("\nImportant fields requiring confirmation:")
print(unavailable_important)

# ------------------------------------------------------------
# 5. Transform and predict for an exploratory test only
# ------------------------------------------------------------
# This test uses the preprocessor's learned imputers for missing
# values. It does NOT make the missing information factual.
X_processed = preprocessor.transform(input_df)

expected_features = len(preprocessor.get_feature_names_out())
if X_processed.shape[1] != expected_features:
    raise ValueError(
        f"Feature mismatch: got {X_processed.shape[1]}, "
        f"expected {expected_features}"
    )

print("\nProcessed shape:", X_processed.shape)

if unavailable_important:
    print(
        "\nPREDICTION NOT DISPLAYED: too many important fields "
        "are unavailable. Confirm the missing fields first."
    )
else:
    predicted_class = int(model.predict(X_processed)[0])
    estimated_score = float(model.predict_proba(X_processed)[0, 1])

    print("\nExploratory model output:")
    print("Predicted class:", predicted_class)
    print("Class meaning:", "Placed" if predicted_class == 1 else "Not placed")
    print(f"Model-estimated score: {estimated_score:.1%}")

    print(
        "\nCaution: this model was trained on synthetic data. "
        "Its score is not a validated real-world placement probability."
    )


RESUME-TO-MODEL INTEGRATION
Resume: Vishali_Resume__1___1_.pdf
Extracted skills: ['C', 'C++', 'CSS', 'Computer Networks', 'Data Structures', 'Deep Learning', 'GitHub', 'HTML', 'Java', 'Machine Learning', 'MongoDB', 'MySQL', 'Node.js', 'Python', 'React', 'SQL']
Expected raw input columns: 153

Extracted CGPA: 8.55
Number of extracted skills: 16

Fields mapped before preprocessing: 20
Fields still missing: 133

Mapped fields:
['Education_Level', 'UG_Degree', 'PG_Specialization', 'Branch', 'CGPA', 'C', 'Cpp', 'Java', 'Python', 'Computer_Networks', 'HTML', 'CSS', 'React', 'NodeJS', 'SQL', 'MySQL', 'MongoDB', 'Machine_Learning', 'Deep_Learning', 'GitHub']

Missing fields (first 50):
['PG_Required_For_Target_Role', 'Relevant_Degree_Required', 'Year', 'Attendance_Percentage', 'Backlogs', 'Core_Subject_Score', 'Technical_Subject_Score', 'Aptitude_Score', 'JavaScript', 'TypeScript', 'CSharp', 'R', 'Go', 'Kotlin', 'PHP', 'Ruby', 'Swift', 'Dart', 'MATLAB', 'DSA', 'OOP', 'DBMS', 'Operating_Systems

In [4]:
# ============================================================
# STEP 3D: COLLECT MISSING STUDENT INFORMATION
# ============================================================

import pandas as pd
import json

resume_path = "/content/resume_extraction_output.json"

with open(resume_path, "r", encoding="utf-8") as f:
    resume_data = json.load(f)

def ask_number(prompt, minimum=None, maximum=None, allow_blank=False):
    while True:
        value = input(prompt).strip()

        if allow_blank and value == "":
            return None

        try:
            number = float(value)

            if minimum is not None and number < minimum:
                print(f"Enter a value >= {minimum}.")
                continue

            if maximum is not None and number > maximum:
                print(f"Enter a value <= {maximum}.")
                continue

            return number

        except ValueError:
            print("Please enter a valid number.")

print("=" * 60)
print("STUDENT INFORMATION")
print("=" * 60)

print("Resume:", resume_data.get("resume_filename", "Unknown"))
print("Extracted CGPA:", resume_data.get("cgpa_mentions", []))

# Confirm education details
education_level = input(
    "Education level (UG/PG): "
).strip()

ug_degree = input(
    "UG degree and branch (e.g., B.Tech IT): "
).strip()

branch = input(
    "Branch (CSE/IT/ECE/EEE/Mechanical/Civil/Aerospace): "
).strip()

year = ask_number(
    "Current year of study (e.g., 3 or 4): ", 1, 8
)

# Internship details
internship_answer = input(
    "Have you completed an internship? (yes/no): "
).strip().lower()

internship = 1 if internship_answer in ["yes", "y"] else 0

if internship == 1:
    internship_duration = ask_number(
        "Internship duration in months: ", 0, 120
    )
else:
    internship_duration = 0

# Project details
project_count = ask_number(
    "Total number of projects: ", 0, 100
)

relevant_project_count = ask_number(
    "Number of projects relevant to your target role: ",
    0, project_count
)

# Academic details
backlogs = ask_number(
    "Current/past backlogs count: ", 0, 100
)

attendance = ask_number(
    "Attendance percentage (press Enter if unknown): ",
    0, 100, allow_blank=True
)

core_score = ask_number(
    "Core-subject score (use the dataset's scale; Enter if unknown): ",
    0, 100, allow_blank=True
)

technical_score = ask_number(
    "Technical-subject score (same scale; Enter if unknown): ",
    0, 100, allow_blank=True
)

aptitude_score = ask_number(
    "Aptitude score (same scale; Enter if unknown): ",
    0, 100, allow_blank=True
)

student_inputs = {
    "Education_Level": education_level,
    "UG_Degree": ug_degree,
    "Branch": branch,
    "Year": year,
    "Internship": internship,
    "Internship_Duration_Months": internship_duration,
    "Project_Count": project_count,
    "Relevant_Project_Count": relevant_project_count,
    "Backlogs": backlogs,
    "Attendance_Percentage": attendance,
    "Core_Subject_Score": core_score,
    "Technical_Subject_Score": technical_score,
    "Aptitude_Score": aptitude_score
}

print("\n" + "=" * 60)
print("COLLECTED INFORMATION")
print("=" * 60)

for field, value in student_inputs.items():
    print(f"{field}: {value}")

print("\nInformation collected. Prediction is not yet run.")

STUDENT INFORMATION
Resume: Vishali_Resume__1___1_.pdf
Extracted CGPA: ['CGPA 8.55']
Education level (UG/PG): ug
UG degree and branch (e.g., B.Tech IT): B.Tech IT
Branch (CSE/IT/ECE/EEE/Mechanical/Civil/Aerospace): IT
Current year of study (e.g., 3 or 4): 4
Have you completed an internship? (yes/no): no
Total number of projects: 4
Number of projects relevant to your target role: 3
Current/past backlogs count: 0
Attendance percentage (press Enter if unknown): 98
Core-subject score (use the dataset's scale; Enter if unknown): 63
Technical-subject score (same scale; Enter if unknown): 78
Aptitude score (same scale; Enter if unknown): 76

COLLECTED INFORMATION
Education_Level: ug
UG_Degree: B.Tech IT
Branch: IT
Year: 4.0
Internship: 0
Internship_Duration_Months: 0
Project_Count: 4.0
Relevant_Project_Count: 3.0
Backlogs: 0.0
Attendance_Percentage: 98.0
Core_Subject_Score: 63.0
Technical_Subject_Score: 78.0
Aptitude_Score: 76.0

Information collected. Prediction is not yet run.


In [5]:
# ============================================================
# STEP 3D: RESUME-BASED EMPLOYABILITY PREDICTION
# User inputs: CGPA and active backlogs ONLY
# ============================================================

import json
import re
import joblib
import numpy as np
import pandas as pd
from pathlib import Path

BASE = "/content"

RESUME_JSON = f"{BASE}/resume_extraction_output.json"
MODEL_PATH = f"{BASE}/final_employability_model.joblib"
PREPROCESSOR_PATH = f"{BASE}/preprocessor.joblib"

# ------------------------------------------------------------
# 1. LOAD FILES
# ------------------------------------------------------------

with open(RESUME_JSON, "r", encoding="utf-8") as f:
    resume = json.load(f)

model = joblib.load(MODEL_PATH)
preprocessor = joblib.load(PREPROCESSOR_PATH)

raw_columns = list(preprocessor.feature_names_in_)

print("=" * 65)
print("RESUME-BASED EMPLOYABILITY PREDICTION")
print("=" * 65)
print("Resume:", resume.get("resume_filename", "Unknown"))
print("Extracted skills:", resume.get("technical_skills", []))
print("Total extracted skills:", len(resume.get("technical_skills", [])))
print("Expected raw input features:", len(raw_columns))

# ------------------------------------------------------------
# 2. ASK ONLY FOR CGPA AND ACTIVE BACKLOGS
# ------------------------------------------------------------

def ask_float(prompt, minimum, maximum, integer=False):
    while True:
        try:
            value = float(input(prompt).strip())

            if not minimum <= value <= maximum:
                print(f"Enter a value between {minimum} and {maximum}.")
                continue

            if integer and not value.is_integer():
                print("Enter a whole number.")
                continue

            return int(value) if integer else value

        except ValueError:
            print("Please enter a valid number.")

cgpa = ask_float("Enter your CGPA (out of 10): ", 0, 10)
active_backlogs = ask_float(
    "Enter your current active backlog count: ",
    0, 100, integer=True
)

# ------------------------------------------------------------
# 3. CREATE A RAW INPUT ROW
# ------------------------------------------------------------

# Unknown profile values remain missing or Unknown.
# The saved preprocessor handles missing numeric values.
row = {column: np.nan for column in raw_columns}

categorical_columns = [
    "Education_Level",
    "UG_Degree",
    "PG_Specialization",
    "Branch"
]

for column in categorical_columns:
    if column in row:
        row[column] = "Unknown"

# If the resume mentions B.Tech or B.E, infer undergraduate level.
education_text = " ".join(
    str(x) for x in resume.get("education_mentions", [])
).lower()

if "Education_Level" in row:
    if any(term in education_text for term in [
        "b.tech", "btech", "b.e", "bachelor", "undergraduate"
    ]):
        row["Education_Level"] = "UG"

# Map only degree details actually supported by extracted education.
# Do not guess a branch or degree specialization.

if "CGPA" in row:
    row["CGPA"] = cgpa

if "Backlogs" in row:
    row["Backlogs"] = active_backlogs

# ------------------------------------------------------------
# 4. IDENTIFY SKILL FEATURES
# ------------------------------------------------------------

# These are profile/academic/experience fields, not technical skills.
non_skill_columns = {
    "Student_ID",
    "Data_Source",
    "Placement_Status",
    "Education_Level",
    "UG_Degree",
    "PG_Specialization",
    "Branch",
    "Year",
    "CGPA",
    "Attendance_Percentage",
    "Backlogs",
    "Core_Subject_Score",
    "Technical_Subject_Score",
    "Aptitude_Score",
    "Internship",
    "Internship_Duration_Months",
    "Project_Count",
    "Relevant_Project_Count",
    "Certification_Count",
    "Relevant_Certification_Count",
    "Open_Source_Contribution",
    "PG_Required_For_Target_Role",
    "Relevant_Degree_Required"
}

# All remaining feature columns are treated as skill indicators.
# This assumes the dataset uses binary columns for these skills.
skill_columns = [
    col for col in raw_columns
    if col not in non_skill_columns
    and col not in categorical_columns
]

# Start every skill at 0: not detected in the resume.
for column in skill_columns:
    row[column] = 0

# ------------------------------------------------------------
# 5. MAP EXTRACTED SKILLS TO DATASET FEATURE NAMES
# ------------------------------------------------------------

def normalize(text):
    return re.sub(r"[^a-z0-9]", "", str(text).lower())

# Alias map: extracted resume terms -> dataset feature names.
aliases = {
    "c++": "Cpp",
    "cpp": "Cpp",
    "node.js": "NodeJS",
    "nodejs": "NodeJS",
    "computer networks": "Computer_Networks",
    "data structures": "DSA",
    "data structures and algorithms": "DSA",
    "machine learning": "Machine_Learning",
    "deep learning": "Deep_Learning",
    "sql": "SQL",
    "github": "GitHub"
}

# Normalized dataset feature lookup.
feature_lookup = {
    normalize(col): col for col in skill_columns
}

# Include explicit aliases where the dataset uses different names.
alias_targets = {
    normalize(alias): target
    for alias, target in aliases.items()
}

extracted_skills = resume.get("technical_skills", [])
matched_skills = []
unmatched_skills = []

for skill in extracted_skills:
    normalized_skill = normalize(skill)

    target = alias_targets.get(normalized_skill)

    if target is None:
        target = feature_lookup.get(normalized_skill)

    if target in skill_columns:
        row[target] = 1
        matched_skills.append((skill, target))
    else:
        unmatched_skills.append(skill)

# ------------------------------------------------------------
# 6. DERIVE PROJECT / CERTIFICATION COUNTS FROM RESUME SECTIONS
# ------------------------------------------------------------

def section_items(section):
    if isinstance(section, list):
        return [x for x in section if str(x).strip()]
    if isinstance(section, str) and section.strip():
        return [section.strip()]
    return []

projects = section_items(resume.get("project_section", []))
certifications = section_items(resume.get("certification_section", []))

if "Project_Count" in row:
    row["Project_Count"] = len(projects)

if "Certification_Count" in row:
    row["Certification_Count"] = len(certifications)

# Do not infer internship from a section that may contain projects.
# Internship fields remain missing if reliable internship data is absent.

# ------------------------------------------------------------
# 7. PREPROCESS AND PREDICT
# ------------------------------------------------------------

X_raw = pd.DataFrame([row], columns=raw_columns)

print("\n" + "=" * 65)
print("SKILL MAPPING RESULTS")
print("=" * 65)
print("Dataset skill features:", len(skill_columns))
print("Matched resume skills:", len(matched_skills))
print("Unmatched extracted skills:", unmatched_skills)
print("\nMatched skills:")
for resume_skill, dataset_feature in matched_skills:
    print(f"  {resume_skill} -> {dataset_feature}")

# Confirm every dataset skill indicator is binary.
skill_values = X_raw[skill_columns].iloc[0]
assert skill_values.isin([0, 1]).all()

# Use the SAME preprocessor fitted during training.
X_processed = preprocessor.transform(X_raw)

print("\nRaw input shape:", X_raw.shape)
print("Processed input shape:", X_processed.shape)

# Guard against model/preprocessor feature mismatch.
expected_features = getattr(model, "n_features_in_", None)
actual_features = X_processed.shape[1]

if expected_features is not None and expected_features != actual_features:
    raise ValueError(
        f"Feature mismatch: model expects {expected_features}, "
        f"but preprocessor produced {actual_features}."
    )

prediction = int(model.predict(X_processed)[0])

print("\n" + "=" * 65)
print("EMPLOYABILITY MODEL OUTPUT")
print("=" * 65)
print("Predicted class:", prediction)
print("Interpretation:", "Placed class (1)" if prediction == 1
      else "Not-placed class (0)")

if hasattr(model, "predict_proba"):
    probabilities = model.predict_proba(X_processed)[0]
    class_labels = list(model.classes_)

    if 1 in class_labels:
        placed_probability = float(probabilities[class_labels.index(1)])
        print(f"Model-estimated placed-class score: "
              f"{placed_probability * 100:.2f}%")

print(
    "\nNote: The score is based on a model trained on synthetic data; "
    "it is not a validated real-world placement probability."
)

RESUME-BASED EMPLOYABILITY PREDICTION
Resume: Vishali_Resume__1___1_.pdf
Extracted skills: ['C', 'C++', 'CSS', 'Computer Networks', 'Data Structures', 'Deep Learning', 'GitHub', 'HTML', 'Java', 'Machine Learning', 'MongoDB', 'MySQL', 'Node.js', 'Python', 'React', 'SQL']
Total extracted skills: 16
Expected raw input features: 153
Enter your CGPA (out of 10): 8.5
Enter your current active backlog count: 0

SKILL MAPPING RESULTS
Dataset skill features: 133
Matched resume skills: 16
Unmatched extracted skills: []

Matched skills:
  C -> Cpp
  C++ -> Cpp
  CSS -> CSS
  Computer Networks -> Computer_Networks
  Data Structures -> DSA
  Deep Learning -> Deep_Learning
  GitHub -> GitHub
  HTML -> HTML
  Java -> Java
  Machine Learning -> Machine_Learning
  MongoDB -> MongoDB
  MySQL -> MySQL
  Node.js -> NodeJS
  Python -> Python
  React -> React
  SQL -> SQL

Raw input shape: (1, 153)
Processed input shape: (1, 169)

EMPLOYABILITY MODEL OUTPUT
Predicted class: 0
Interpretation: Not-placed clas

In [6]:
# Inspect the feature mapping and raw resume input

import numpy as np
import pandas as pd
import joblib

preprocessor = joblib.load("/content/preprocessor.joblib")
model = joblib.load("/content/final_employability_model.joblib")

print("Raw input columns:", len(preprocessor.feature_names_in_))
print("Model input features:", model.n_features_in_)

# Display the preprocessor's numeric and categorical columns
for name, transformer, columns in preprocessor.transformers_:
    if name != "remainder":
        print(f"\n{name}: {len(columns)} columns")
        print(list(columns)[:30])

print("\nCheck whether separate C and C++ features exist:")
raw_cols = set(preprocessor.feature_names_in_)
print("C exists:", "C" in raw_cols)
print("Cpp exists:", "Cpp" in raw_cols)

# Check the fitted numeric imputation strategy
print("\nNumeric pipeline:")
print(preprocessor.named_transformers_["num"])

Raw input columns: 153
Model input features: 169

num: 149 columns
['PG_Required_For_Target_Role', 'Relevant_Degree_Required', 'Year', 'CGPA', 'Attendance_Percentage', 'Backlogs', 'Core_Subject_Score', 'Technical_Subject_Score', 'Aptitude_Score', 'C', 'Cpp', 'Java', 'Python', 'JavaScript', 'TypeScript', 'CSharp', 'R', 'Go', 'Kotlin', 'PHP', 'Ruby', 'Swift', 'Dart', 'MATLAB', 'DSA', 'OOP', 'DBMS', 'Operating_Systems', 'Computer_Networks', 'Computer_Architecture']

cat: 4 columns
['Education_Level', 'UG_Degree', 'PG_Specialization', 'Branch']

Check whether separate C and C++ features exist:
C exists: True
Cpp exists: True

Numeric pipeline:
Pipeline(steps=[('imputer', SimpleImputer(strategy='median'))])
